In [2]:
import importlib.util
import subprocess
import sys

required = ["pandas", "pygwalker", "pyodbc", "dotenv", "sqlalchemy"]
missing = [package for package in required if importlib.util.find_spec(package) is None]

if missing:
    print(f"Paquetes faltantes en el kernel actual: {missing}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet",
         "pandas", "pygwalker", "sqlalchemy", "pyodbc", "python-dotenv"]
    )
    print("Instalación completada.")
else:
    print("Todos los paquetes requeridos ya están disponibles en este kernel.")


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import importlib
import os
from pathlib import Path

missing = []
for module_name in ["pandas", "pygwalker", "pyodbc", "dotenv", "sqlalchemy"]:
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing.append(module_name)

if missing:
    print(f"Instalando dependencias faltantes: {missing}")
    %pip install --quiet pandas pygwalker sqlalchemy pyodbc python-dotenv
    for module_name in missing:
        importlib.import_module(module_name)

# VS Code/pyright puede marcar estas importaciones como no resueltas
# si el kernel activo no coincide con el entorno donde se instalaron.
# Se ignora el warning específico de resolución del módulo para no bloquear la ejecución.
import pandas as pd  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
import pygwalker as pyg  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
import pyodbc  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
from dotenv import load_dotenv  # pyright: ignore[reportMissingImports, reportMissingModuleSource]
from sqlalchemy import create_engine, text  # pyright: ignore[reportMissingImports, reportMissingModuleSource]

# 1. Carga explícita y robusta del entorno
base_dir = Path.cwd().resolve()
env_path = None
for candidate in [
    base_dir / ".env",
    base_dir.parent / ".env",
    base_dir / ".." / ".env",
    Path("../.env"),
]:
    resolved = candidate.resolve()
    if resolved.is_file():
        env_path = resolved
        break

if env_path is None:
    raise RuntimeError(
        "❌ No se encontró el archivo .env en la raíz esperada. "
        "Revisa que exista en P4_Real_World_Ingestion/.env"
    )

# No forzamos autenticación de Windows cuando no se especifica de forma explícita.
for key in ["DB_USER", "DB_PASSWORD", "DB_USE_TRUSTED_CONNECTION"]:
    os.environ.pop(key, None)

load_dotenv(dotenv_path=env_path, override=True)

print(f"✅ Variables de entorno cargadas desde: {env_path}")

server = (os.getenv("DB_SERVER") or "").strip()
database = (os.getenv("DB_NAME") or "").strip()
driver = (os.getenv("DB_DRIVER") or "ODBC Driver 17 for SQL Server").strip()
db_user = (os.getenv("DB_USER") or "").strip()
db_password = (os.getenv("DB_PASSWORD") or "").strip()
use_trusted = os.getenv("DB_USE_TRUSTED_CONNECTION", "false").strip().lower() in {"1", "true", "yes", "y"}

if (not db_user or not db_password) and not use_trusted:
    use_trusted = True
    print("ℹ️ No se detectaron DB_USER/DB_PASSWORD; se usará autenticación de Windows por defecto.")

if not server or not database:
    raise RuntimeError(
        "❌ Faltan credenciales críticas (DB_SERVER o DB_NAME). Revisa tu archivo .env."
    )

# Valida que el driver exista y usa un fallback si el 17 no está instalado.
available_drivers = [d for d in pyodbc.drivers()]
if driver not in available_drivers:
    fallback_drivers = ["ODBC Driver 18 for SQL Server", "ODBC Driver 17 for SQL Server", "SQL Server"]
    for candidate_driver in fallback_drivers:
        if candidate_driver in available_drivers:
            driver = candidate_driver
            break

if driver not in available_drivers:
    raise RuntimeError(
        "❌ No se encontró un driver ODBC de SQL Server instalado. "
        "Instala 'ODBC Driver 17 for SQL Server' o 'ODBC Driver 18 for SQL Server'.\n"
        f"Drivers disponibles: {available_drivers}"
    )

# Construcción segura de la cadena de conexión.
conn_parts = [
    f"DRIVER={{{driver}}}",
    f"SERVER={server}",
    f"DATABASE={database}",
    "Encrypt=yes",
    "TrustServerCertificate=yes",
    "Connection Timeout=30",
]

if db_user and db_password:
    conn_parts.append(f"UID={db_user}")
    conn_parts.append(f"PWD={db_password}")
    auth_mode = "SQL Server Authentication"
elif use_trusted:
    conn_parts.append("Trusted_Connection=yes")
    auth_mode = "Windows Authentication"
else:
    raise RuntimeError(
        "❌ Faltan credenciales. Define DB_USER y DB_PASSWORD para autenticación SQL, "
        "o habilita DB_USE_TRUSTED_CONNECTION=true para autenticación de Windows en el .env."
    )

conn_string = ";".join(conn_parts) + ";"
print(f"🔐 Intentando conectar con {auth_mode} a {server}/{database} usando driver {driver}...")

# Crea el motor SQLAlchemy usando pyodbc con la cadena de conexión ya validada.
engine = create_engine(
    "mssql+pyodbc://",
    creator=lambda: pyodbc.connect(conn_string, timeout=30, autocommit=True),
)

# 2. Validación de conexión con diagnóstico útil
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@SERVERNAME AS server_name, DB_NAME() AS current_db")).fetchone()
        print("✅ Conexión a SQL Server correcta.")
        print(f"Servidor/BD actual: {result}")
except Exception as e:
    if use_trusted:
        current_user = os.getlogin() if os.name == "nt" else "el usuario actual del sistema"
        user_hint = (
            f"Si usas autenticación de Windows, revisa que '{current_user}' exista como Login "
            "en SQL Server y tenga permisos sobre la base de datos. "
            f"Además confirma que la base de datos '{database}' exista en el servidor '{server}'."
        )
    else:
        user_hint = (
            f"Revisa que el usuario '{db_user}' exista como Login en SQL Server, que la contraseña sea correcta "
            f"y que el usuario tenga permisos sobre la base de datos '{database}'."
        )

    print(
        "⚠️ No se pudo establecer la conexión a SQL Server. "
        "Se usará un conjunto de datos de ejemplo para que la dashboard siga funcionando.\n"
        f"Detalle técnico: {e}\n"
        f"Sugerencia: {user_hint}"
    )
    df_logistics = pd.DataFrame(
        {
            "OrderID": [1001, 1002, 1003, 1004, 1005],
            "Region": ["North", "South", "East", "West", "Central"],
            "Carrier": ["DHL", "FedEx", "UPS", "DHL", "FedEx"],
            "ShippingMode": ["Air", "Sea", "Ground", "Air", "Ground"],
            "ShipDate": pd.to_datetime(["2024-01-04", "2024-01-06", "2024-01-08", "2024-01-09", "2024-01-10"]),
            "DeliveryDays": [2, 6, 4, 3, 5],
            "OnTimeRate": [0.96, 0.88, 0.93, 0.91, 0.89],
            "CostUSD": [1400, 980, 600, 1100, 720],
        }
    )
    df_performance = pd.DataFrame(
        {
            "Category": ["Electronics", "Home", "Fashion", "Office", "Health"],
            "SalesUSD": [240000, 180000, 140000, 170000, 210000],
            "Units": [1200, 950, 1100, 750, 980],
            "ProfitMargin": [0.28, 0.24, 0.19, 0.22, 0.27],
            "Region": ["North", "West", "East", "South", "Central"],
        }
    )
else:
    # 3. Consumo de vistas analíticas
    print("📊 Extrayendo datos desde SQL Server...")
    try:
        df_logistics = pd.read_sql(text("SELECT * FROM Analytics.vw_Shipping_Efficiency"), engine)
        df_performance = pd.read_sql(text("SELECT * FROM Analytics.vw_Category_Performance"), engine)
    except Exception as e:
        print(
            "⚠️ La conexión a SQL Server sí funciona, pero las vistas analíticas no están disponibles. "
            "Se usará un conjunto de datos de ejemplo para continuar.\n"
            f"Detalle técnico: {e}"
        )
        df_logistics = pd.DataFrame(
            {
                "OrderID": [1001, 1002, 1003, 1004, 1005],
                "Region": ["North", "South", "East", "West", "Central"],
                "Carrier": ["DHL", "FedEx", "UPS", "DHL", "FedEx"],
                "ShippingMode": ["Air", "Sea", "Ground", "Air", "Ground"],
                "ShipDate": pd.to_datetime(["2024-01-04", "2024-01-06", "2024-01-08", "2024-01-09", "2024-01-10"]),
                "DeliveryDays": [2, 6, 4, 3, 5],
                "OnTimeRate": [0.96, 0.88, 0.93, 0.91, 0.89],
                "CostUSD": [1400, 980, 600, 1100, 720],
            }
        )
        df_performance = pd.DataFrame(
            {
                "Category": ["Electronics", "Home", "Fashion", "Office", "Health"],
                "SalesUSD": [240000, 180000, 140000, 170000, 210000],
                "Units": [1200, 950, 1100, 750, 980],
                "ProfitMargin": [0.28, 0.24, 0.19, 0.22, 0.27],
                "Region": ["North", "West", "East", "South", "Central"],
            }
        )

print(f"✅ Filas recuperadas (Logística): {len(df_logistics)}")
print(f"✅ Filas recuperadas (Desempeño): {len(df_performance)}")

# 4. Lanzamiento de interfaz interactiva
if not df_logistics.empty:
    print("🚀 Lanzando interfaz interactiva...")
    walker = pyg.walk(df_logistics)
else:
    print("❌ ERROR: El DataFrame de logística está vacío. Revisa la vista en SQL.")


✅ Variables de entorno cargadas desde: D:\01.Datos_AlbertoDzib\Portafolio-Alberto\P4_Real_World_Ingestion\.env
ℹ️ No se detectaron DB_USER/DB_PASSWORD; se usará autenticación de Windows por defecto.
🔐 Intentando conectar con Windows Authentication a DZIBPC/P4_Global_SupplyChain usando driver ODBC Driver 17 for SQL Server...
⚠️ No se pudo establecer la conexión a SQL Server. Se usará un conjunto de datos de ejemplo para que la dashboard siga funcionando.
Detalle técnico: (pyodbc.InterfaceError) ('28000', '[28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'DzibPC\\dzibj\'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot open database "P4_Global_SupplyChain" requested by the login. The login failed. (4060); [28000] [Microsoft][ODBC Driver 17 for SQL Server]Invalid connection string attribute (0); [28000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Login failed for user \'DzibPC\\dzibj\'. (18456);

Box(children=(HTML(value='\n<div id="ifr-pyg-00065d0d6cae9bf6s2jDuXgCAU8FVK1B" style="height: auto">\n    <hea…

In [8]:
from pathlib import Path
p = Path(r"D:\01.Datos_AlbertoDzib\Portafolio-Alberto\P4_Real_World_Ingestion\.env")
print(p.read_text(encoding="utf-8"))


DB_SERVER=DZIBPC
DB_NAME=P4_Global_SupplyChain
DB_DRIVER=ODBC Driver 17 for SQL Server

